In [226]:
import pandas as pd

In [227]:
df_wdi = pd.read_excel('data/P_Data_Extract_From_World_Development_Indicators.xlsx')

In [228]:
df_wdi.replace('..', pd.NA, inplace=True)

In [229]:
df_wdi.drop(range(148,153), inplace=True)
df_wdi.drop(["Series Code"], axis=1, inplace=True)

In [230]:
df_wdi.head(2)

,Country Name,Country Code,Series Name,1960 [YR1960],1961 [YR1961],1962 [YR1962],1963 [YR1963],1964 [YR1964],1965 [YR1965],1966 [YR1966],...,2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024],2025 [YR2025]
0,Austria,AUT,"School enrollment, primary (% net)",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,88.84197,88.61718,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,Austria,AUT,Individuals using the Internet (% of population),<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,84.3237,87.9356,87.4791,87.7522,87.5294,92.5292,93.6141,95.3347,94.9197,<NA>


In [231]:
df_wdi["Series Name"].unique()

array(['School enrollment, primary (% net)',
       'Individuals using the Internet (% of population)',
       'Life expectancy at birth, total (years)',
       'GDP per capita, PPP (current international $)', nan], dtype=object)

In [232]:
# Extract year from column names and rename
year_columns = [col for col in df_wdi.columns if '[YR' in col]
for col in year_columns:
    year = col.split('[YR')[1].split(']')[0]
    df_wdi.rename(columns={col: year}, inplace=True)

In [233]:
temp_enroll, temp_internet, temp_le, temp_gdp,_ = (df_wdi[df_wdi["Series Name"]==x] for x in df_wdi["Series Name"].unique())

In [234]:
temp_enroll=temp_enroll.melt(id_vars=["Country Name", "Country Code", "Series Name"], var_name="Year", value_name="School enrollment, primary (% net)")
temp_internet=temp_internet.melt(id_vars=["Country Name", "Country Code", "Series Name"], var_name="Year", value_name="Individuals using the Internet (% of population)")
temp_le=temp_le.melt(id_vars=["Country Name", "Country Code", "Series Name"], var_name="Year", value_name="Life expectancy at birth, total (years)")
temp_gdp=temp_gdp.melt(id_vars=["Country Name", "Country Code", "Series Name"], var_name="Year", value_name="GDP per capita (current international $)")

In [235]:
temp_enroll['Year'] = temp_enroll['Year'].astype(int)
temp_internet['Year'] = temp_internet['Year'].astype(int)
temp_le['Year'] = temp_le['Year'].astype(int)
temp_gdp['Year'] = temp_gdp['Year'].astype(int)

In [236]:
temp_enroll.drop('Series Name', axis=1, inplace=True)
temp_internet.drop('Series Name', axis=1, inplace=True)
temp_le.drop('Series Name', axis=1, inplace=True)
temp_gdp.drop('Series Name', axis=1, inplace=True)

In [237]:
temp_enroll['School enrollment, primary (% net)'] = temp_enroll.groupby('Country Code').ffill()['School enrollment, primary (% net)']
temp_internet['Individuals using the Internet (% of population)'] = temp_internet.groupby('Country Code').ffill()['Individuals using the Internet (% of population)']
temp_le['Life expectancy at birth, total (years)'] = temp_le.groupby('Country Code').ffill()['Life expectancy at birth, total (years)']
temp_gdp['GDP per capita (current international $)'] = temp_gdp.groupby('Country Code').ffill()['GDP per capita (current international $)']

In [238]:
df_wdi_final = pd.merge(temp_enroll, temp_internet, on=["Country Name", "Country Code", "Year"], how="inner") \
                 .merge(temp_le, on=["Country Name", "Country Code", "Year"], how="inner") \
                 .merge(temp_gdp, on=["Country Name", "Country Code", "Year"], how="inner")

In [239]:
df_wdi_final.head(2)

,Country Name,Country Code,Year,"School enrollment, primary (% net)",Individuals using the Internet (% of population),"Life expectancy at birth, total (years)",GDP per capita (current international $)
0,Austria,AUT,1960,NaN,NaN,68.58561,NaN
1,Albania,ALB,1960,NaN,NaN,56.413,NaN


In [240]:
import country_converter as cc

In [241]:
df_whr = pd.read_excel('data/WHR26_Data_Figure_2.1.xlsx')

In [242]:
df_whr.head()

,Year,Rank,Country name,Life evaluation (3-year average),Lower whisker,Upper whisker,Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,Dystopia + residual
0,2025,1,Finland,7.764,7.690,7.837,1.915,1.638,0.939,1.105,0.093,0.491,1.582
1,2025,2,Iceland,7.540,7.449,7.630,1.971,1.720,0.996,1.105,0.187,0.187,1.373
2,2025,3,Denmark,7.539,7.446,7.631,1.986,1.633,0.930,1.081,0.125,0.474,1.310
3,2025,4,Costa Rica,7.439,7.356,7.522,1.697,1.483,0.739,1.101,0.059,0.122,2.236
4,2025,5,Sweden,7.255,7.172,7.337,1.950,1.570,1.027,1.070,0.149,0.447,1.041


In [243]:
df_whr['Country Code'] = cc.convert(names=df_whr['Country name'], to='ISO3')

In [244]:
atributi = [
    "Year",
    "Country Code", 
    'Life evaluation (3-year average)', 
    'Explained by: Log GDP per capita', 
    'Explained by: Social support', 
    'Explained by: Healthy life expectancy', 
    'Explained by: Freedom to make life choices', 
    'Explained by: Generosity', 
    'Explained by: Perceptions of corruption',
    'Dystopia + residual'
]

skupno = pd.merge(df_wdi_final, df_whr[atributi], on=["Year", "Country Code"], how="inner")

In [245]:
skupno.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 658 entries, 0 to 657
Data columns (total 15 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   Country Name                                      658 non-null    object 
 1   Country Code                                      658 non-null    object 
 2   Year                                              658 non-null    int64  
 3   School enrollment, primary (% net)                620 non-null    object 
 4   Individuals using the Internet (% of population)  653 non-null    object 
 5   Life expectancy at birth, total (years)           658 non-null    object 
 6   GDP per capita (current international $)          658 non-null    object 
 7   Life evaluation (3-year average)                  658 non-null    float64
 8   Explained by: Log GDP per capita                  322 non-null    float64
 9   Explained by: Social 

In [246]:
skupno.head(2)

,Country Name,Country Code,Year,"School enrollment, primary (% net)",Individuals using the Internet (% of population),"Life expectancy at birth, total (years)",GDP per capita (current international $),Life evaluation (3-year average),Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,Dystopia + residual
0,Austria,AUT,2011,87.15916,78.74,80.982927,44171.561869,7.227,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Albania,ALB,2011,86.99052,47,78.303,10273.419978,5.134,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [247]:
df_hdi = pd.read_csv('data/HDR25_Composite_indices_complete_time_series.csv', encoding='Windows-1250')

In [248]:
df_hdi.head(2)

,iso3,country,hdicode,region,hdi_rank_2023,hdi_1990,hdi_1991,hdi_1992,hdi_1993,hdi_1994,...,pop_total_2014,pop_total_2015,pop_total_2016,pop_total_2017,pop_total_2018,pop_total_2019,pop_total_2020,pop_total_2021,pop_total_2022,pop_total_2023
0,AFG,Afghanistan,Low,SA,181.0,0.285,0.291,0.301,0.311,0.305,...,32.792523,33.831764,34.700612,35.688935,36.743039,37.856121,39.068979,40.000412,40.578842,41.454761
1,ALB,Albania,Very High,ECA,71.0,0.654,0.638,0.622,0.624,0.629,...,2.903749,2.898632,2.897867,2.898242,2.894231,2.885009,2.871954,2.849636,2.827608,2.811655


In [249]:
df_hdi = pd.wide_to_long(df_hdi, stubnames=['hdi','le','eys','mys','gnipc'], i='iso3', j='Year', sep='_')

In [250]:
df_hdi.reset_index(inplace=True)

In [251]:
df_hdi.head(2)

,iso3,Year,abr_1990,abr_1991,abr_1992,abr_1993,abr_1994,abr_1995,abr_1996,abr_1997,...,se_m_2019,se_m_2020,se_m_2021,se_m_2022,se_m_2023,hdi,le,eys,mys,gnipc
0,AFG,1990,139.376,145.383,147.499,149.461,156.835,158.315,157.603,158.761,...,25.989892,26.026215,14.870000,24.077040,24.077040,0.285,45.118,2.93646,0.871962,3642.049616
1,ALB,1990,15.828,15.684,17.235,19.007,20.968,21.862,15.129,19.295,...,92.677055,93.141407,93.605759,93.857521,93.857521,0.654,72.710,11.57934,7.351565,5622.324727


In [252]:
df_hdi = df_hdi[['iso3', 'Year', 'hdi', 'le', 'eys', 'mys', 'gnipc']]

In [269]:
df_hdi.head(3)

,Country Code,Year,HDI,Life Expectancy,Expected Years of Education,Mean Years of Education,Gross National Income per Capita
0,AFG,1990,0.285,45.118,2.93646,0.871962,3642.049616
1,ALB,1990,0.654,72.710,11.57934,7.351565,5622.324727
2,DZA,1990,0.595,67.658,9.79789,3.917092,11297.324280


In [254]:
df_hdi.rename(columns={'iso3': 'Country Code', 'hdi': 'HDI', 'le': 'Life Expectancy', 'eys': 'Expected Years of Education', 'mys': 'Mean Years of Education', 'gnipc': 'Gross National Income per Capita'}, inplace=True)

In [255]:
df_hdi.head(2)

,Country Code,Year,HDI,Life Expectancy,Expected Years of Education,Mean Years of Education,Gross National Income per Capita
0,AFG,1990,0.285,45.118,2.93646,0.871962,3642.049616
1,ALB,1990,0.654,72.710,11.57934,7.351565,5622.324727


In [ ]:
df_hdi_ranked = df_hdi.copy()
df_hdi_ranked['HDI_Rank'] = df_hdi_ranked.groupby('Year')['HDI'].rank(method='min', ascending=False).astype(int)

In [256]:
skupno = pd.merge(skupno, df_hdi, on=['Country Code', 'Year'], how='inner')

In [257]:
skupno.head(2)

,Country Name,Country Code,Year,"School enrollment, primary (% net)",Individuals using the Internet (% of population),"Life expectancy at birth, total (years)",GDP per capita (current international $),Life evaluation (3-year average),Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,Dystopia + residual,HDI,Life Expectancy,Expected Years of Education,Mean Years of Education,Gross National Income per Capita
0,Austria,AUT,2011,87.15916,78.74,80.982927,44171.561869,7.227,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.915,80.828,16.17313,11.71,62981.65807
1,Albania,ALB,2011,86.99052,47,78.303,10273.419978,5.134,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.781,78.303,14.49334,9.80,12312.54001


In [220]:
skupno_clean = skupno.dropna().copy().reset_index(drop=True)

In [222]:
skupno_clean.head(2)

,Country Name,Country Code,Year,"School enrollment, primary (% net)",Individuals using the Internet (% of population),"Life expectancy at birth, total (years)",GDP per capita (current international $),Life evaluation (3-year average),Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,Dystopia + residual,HDI,Life Expectancy,Expected Years of Education,Mean Years of Education,Gross National Income per Capita
0,Austria,AUT,2019,88.61718,87.7522,81.895122,60354.927942,7.2942,1.317286,1.437445,1.000934,0.603369,0.255510,0.281256,2.398446,0.928,81.907,16.07625,12.260000,65534.68727
1,Albania,ALB,2019,94.52966,68.5504,79.467,16442.125939,4.8827,0.906653,0.830484,0.846330,0.461946,0.171028,0.025361,1.640897,0.805,79.467,15.04347,10.072996,15013.37420


In [258]:
redundant_column = [
    "Life expectancy at birth, total (years)",
    "GDP per capita (current international $)",
    "Life Expectancy",
    "Gross National Income per Capita"
]
skupno_clean.drop(columns=redundant_column, inplace=True)

Poskusi iskanja najboljše vizualizacije za rast...

In [ ]:
#Pogledamo prvo rast
df = skupno_clean.copy()
df = df.sort_values(["Country", "Year"])
df.head()

In [ ]:
growth = df.groupby("Country")["Quality of Life Index"].agg(lambda x: x.iloc[-1] - x.iloc[0])

growth_sorted = growth.sort_values(ascending=False)

top_growth = growth_sorted.head(10)
top_growth

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

top_growth.sort_values().plot(kind='barh')

plt.title("Top 10 Countries with Highest Growth in Quality of Life")
plt.xlabel("Growth (Difference in Index)")
plt.ylabel("Country")

plt.show()

Za analizo rasti kakovosti življenja smo izračunali razliko med zadnjim in prvim letom za vsako državo.
Na grafu vidimo države z največjo rastjo indeksa kakovosti življenja.

In [ ]:
#kombiniramo rast z ostalim 
growth_df = growth.reset_index()
growth_df.columns = ["Country", "Growth"]

avg_features = df.groupby("Country").mean(numeric_only=True).reset_index()

growth_analysis = pd.merge(growth_df, avg_features, on="Country")

growth_analysis.head()

In [ ]:
corr_growth = growth_analysis.corr(numeric_only=True)

corr_growth["Growth"].sort_values(ascending=False)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Priprava podatkov
df_pivot = skupno.pivot_table(index='Country', columns='Year', values='Quality of Life Index')
first_yr, last_yr = df_pivot.columns.min(), df_pivot.columns.max()

# Izračun neto rasti
df_growth = pd.DataFrame(df_pivot[last_yr] - df_pivot[first_yr], columns=['Growth'])
df_growth = df_growth.sort_values(by='Growth', ascending=True) # Razvrstimo od najmanjše do največje

# 2. Ročna določitev barv glede na kvartile (da opis vedno drži)
# Spodnjih 25% = Rdeča, Srednjih 50% = Modra, Zgornjih 25% = Zelena
low_threshold = df_growth['Growth'].quantile(0.25)
high_threshold = df_growth['Growth'].quantile(0.75)

def assign_color(row):
    if row['Growth'] >= high_threshold: return '#2ecc71' # Zelena (Visoka rast)
    elif row['Growth'] <= low_threshold: return '#e74c3c' # Rdeča (Padec/Stagnacija)
    else: return '#3498db' # Modra (Stabilnost)

df_growth['Color'] = df_growth.apply(assign_color, axis=1)

# 3. Izris grafa
plt.figure(figsize=(12, 14))
bars = plt.barh(df_growth.index, df_growth['Growth'], color=df_growth['Color'], edgecolor='black', alpha=0.8)

# Dodatki za preglednost
plt.title(f'Analiza rasti kakovosti življenja v Evropi ({first_yr} - {last_yr})', fontsize=16, pad=20)
plt.xlabel('Sprememba indeksa (točke)', fontsize=12)
plt.ylabel('Država', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)

# Legenda, ki se dejansko ujema z barvami
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', label='Visoka rast (Zgornja četrtina)'),
    Patch(facecolor='#3498db', label='Zmerna rast / Stabilno'),
    Patch(facecolor='#e74c3c', label='Počasna rast / Stagnacija')
]
plt.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

Zakaj graf izgleda tako?
Nizka štartna osnova: Švica ima indeks npr. 200. Da bi zrasla za 100 točk, bi morala postati utopija. Bolgarija je začela pri npr. 60 točkah. Za njo je skok na 140 točk (rast za 80) realen zaradi hitre digitalizacije in rasti plač v zadnjem desetletju.

Stropni efekt: Nemčija in Švica sta že leta 2014 imeli vrhunsko infrastrukturo. Tam ni več prostora za "ekstremno" rast, kvečjemu za vzdrževanje nivoja.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Priprava podatkov
# Izberemo nekaj ključnih držav iz vsake skupine, da graf ne bo neberljiv, 
# ali pa prikažemo vse v mreži (FacetGrid)
evropa_drzave = skupno['Country'].unique()

# Nastavimo stil
sns.set_theme(style="whitegrid")

# 2. Kreiranje FacetGrid - vsaka država svoj graf na časovnici
g = sns.FacetGrid(skupno, col="Country", col_wrap=5, height=3, aspect=1.2)
g.map(sns.lineplot, "Year", "Quality of Life Index", marker="o", color="royalblue", linewidth=2)

# Dodamo referenčno črto (povprečje vseh), da vidiš, kdo je nad/pod povprečjem
avg_qol = skupno.groupby('Year')['Quality of Life Index'].transform('mean')
g.map(plt.axhline, y=skupno['Quality of Life Index'].mean(), color=".7", dashes=(2, 1), zorder=0)

# Oblikovanje naslovov in osi
g.set_axis_labels("Leto", "QoL Index")
g.set_titles("{col_name}")
g.tight_layout()

plt.subplots_adjust(top=0.9)
g.fig.suptitle('Časovna pot kakovosti življenja po državah (2014 - 2025)', fontsize=20)

plt.show()

V tem ključnem delu analize smo vizualizirali razvoj kakovosti življenja skozi celotno časovno obdobje od leta 2014 do 2025. Uporabili smo metodo fasetnih grafov, ki nam omogoča neposredno primerjavo razvojnih poti posameznih evropskih držav na lastnih časovnicah.

Razlaga časovnih trendov:

Zahodna Evropa (Švica, Nemčija, Norveška): Te države kažejo izjemno stabilne, skoraj vodoravne črte na vrhu grafa. To pomeni, da so že pred desetletjem dosegle visok standard, ki ga uspešno vzdržujejo. Nizka rast (ki smo jo videli v prejšnjem grafu) tukaj postane logična – so že pri "stropu" razvitosti.

Vzhodna in Jugovzhodna Evropa (Bolgarija, Romunija, Srbija, Rusija): Pri teh državah opazimo strme naraščajoče krivulje. To so države, ki so v tem obdobju doživele najhitrejšo transformacijo. Njihov "skok" na prejšnjem grafu je tukaj prikazan kot postopen proces prehoda iz nižjega standarda proti evropskemu povprečju.

Vpliv kriznih obdobij: Na časovnicah so jasno vidna nihanja v letih okoli 2020 in 2022, kar sovpada z globalno pandemijo in energetsko krizo. Nekatere države so si opomogle hitreje, kar se odraža v strmini njihove linije proti letu 2025.

S to vizualizacijo smo dokazali, da kakovost življenja ni statična vrednost, temveč dinamičen proces, kjer manj razvite države hitro zmanjšujejo razkorak (t.i. convergence effect), medtem ko razvite države investirajo predvsem v ohranjanje doseženih standardov.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Priprava podatkovne strukture za analizo rasti skozi čas
df_pivot = skupno.pivot_table(index='Country', columns='Year', values='Quality of Life Index')
df_pivot['Net_Growth'] = df_pivot[2025] - df_pivot[2014] # Uporabimo zadnje realno leto ali napoved

# Normalizacija podatkov s pomočjo Z-score (Week 4)
# S tem zagotovimo, da ekstremne vrednosti ne popačijo gručenja
scaler = StandardScaler()
df_growth_clean = df_pivot[['Net_Growth']].dropna()
growth_scaled = scaler.fit_transform(df_growth_clean)

# Nenadzorovano modeliranje: K-means gručenje (Week 5)
# Države razdelimo v 3 skupine: Hitra rast, Stabilni, Nizka rast/Stagnacija
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_growth_clean['Cluster'] = kmeans.fit_predict(growth_scaled)

# Priprava vizualizacije celotnega časovnega trenda za vse države (Week 3)
plt.figure(figsize=(16, 12))
sns.set_style("whitegrid")

# Razvrstimo države po rasti za lepši vizualni učinek
sorted_countries = df_growth_clean.sort_values('Net_Growth', ascending=False).index

# Izris linij za vsako državo posebej v barvah njihovih gruč
palette = {0: '#3498db', 1: '#2ecc71', 2: '#e74c3c'} # Modra, Zelena, Rdeča
for country in sorted_countries:
    country_data = skupno[skupno['Country'] == country]
    cluster_id = df_growth_clean.loc[country, 'Cluster']
    plt.plot(country_data['Year'], country_data['Quality of Life Index'], 
             marker='o', markersize=4, linewidth=2, 
             color=palette[cluster_id], alpha=0.7)
    
    # Dodajanje oznak na konec linij za prvih 10 in zadnjih 10 držav
    if country in sorted_countries[:5] or country in sorted_countries[-5:]:
        plt.text(2025.2, country_data[country_data['Year'] == 2025]['Quality of Life Index'].values[0], 
                 country, fontsize=9, fontweight='bold', color=palette[cluster_id])

plt.title('Dinamika razvoja kakovosti življenja v Evropi (2014-2025)', fontsize=18, pad=20)
plt.xlabel('Leto', fontsize=14)
plt.ylabel('Indeks kakovosti življenja (QoL)', fontsize=14)
plt.xticks(range(2014, 2026))
plt.grid(True, linestyle='--', alpha=0.6)

# Ročna legenda za gruče
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='#2ecc71', lw=3, label='Visoka rast (Rising Stars)'),
    Line2D([0], [0], color='#3498db', lw=3, label='Stabilna rast (Steady Growth)'),
    Line2D([0], [0], color='#e74c3c', lw=3, label='Stagnacija / Relativni upad')
]
plt.legend(handles=legend_elements, loc='upper left', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. Priprava podatkov za izračun rasti
# Pivotiramo tabelo, da dobimo leta kot stolpce
df_pivot = skupno.pivot_table(index='Country', columns='Year', values='Quality of Life Index')

# Izračunamo neto rast med letoma 2014 in 2025
# (Uporabimo .dropna(), da odstranimo države, ki nimajo obeh let)
df_growth = pd.DataFrame(df_pivot[2025] - df_pivot[2014], columns=['Net_Growth']).dropna()

# 2. Gručenje na podlagi rasti (Week 5: K-means)
scaler = StandardScaler()
growth_scaled = scaler.fit_transform(df_growth[['Net_Growth']])

# Določimo 3 skupine (Visoka rast, Stabilna rast, Stagnacija)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_growth['Cluster'] = kmeans.fit_predict(growth_scaled)

# Razvrstimo gruče, da bodo imele logična imena glede na povprečno rast
cluster_means = df_growth.groupby('Cluster')['Net_Growth'].mean().sort_values()
cluster_mapping = {
    cluster_means.index[2]: 'Visoka rast',
    cluster_means.index[1]: 'Stabilna rast',
    cluster_means.index[0]: 'Stagnacija ali padec'
}
df_growth['Category'] = df_growth['Cluster'].map(cluster_mapping)

# 3. Združevanje z vzročnimi dejavniki (WHR faktorji "Explained by")
# Ker so v teh stolpcih tudi NaNi, vzamemo povprečje teh faktorjev skozi leta za vsako državo
factors = [
    'Explained by: Log GDP per capita', 
    'Explained by: Social support', 
    'Explained by: Healthy life expectancy', 
    'Explained by: Freedom to make life choices'
]
df_factors_avg = skupno.groupby('Country')[factors].mean()
df_final_analysis = df_growth.join(df_factors_avg)

#VIZUALIZACIJA 1: Neto sprememba po državah
plt.figure(figsize=(12, 12))
sns.set_style("whitegrid")
df_sorted = df_growth.sort_values('Net_Growth')
colors = {'Visoka rast': '#2ecc71', 'Stabilna rast': '#3498db', 'Stagnacija ali padec': '#e74c3c'}

sns.barplot(
    x=df_sorted['Net_Growth'], 
    y=df_sorted.index, 
    hue=df_sorted['Category'], 
    palette=colors, 
    dodge=False
)
plt.title('Neto sprememba kakovosti življenja po državah (2014-2025)', fontsize=16)
plt.xlabel('Razlika v indeksu (točke)', fontsize=12)
plt.ylabel('Država', fontsize=12)
plt.legend(title='Kategorija rasti', loc='lower right')
plt.tight_layout()
plt.show()

#VIZUALIZACIJA 2: Analiza dejavnikov "ZAKAJ?"
#Izračunamo povprečje WHR faktorjev za vsako kategorijo rasti
df_category_analysis = df_final_analysis.groupby('Category')[factors].mean()

ax = df_category_analysis.T.plot(kind='bar', figsize=(14, 7), color=['#3498db', '#e74c3c', '#2ecc71'])
plt.title('Primerjava ključnih dejavnikov po skupinah rasti', fontsize=16)
plt.ylabel('Povprečen prispevek k sreči/kakovosti', fontsize=12)
plt.xlabel('Dejavniki (WHR "Explained by")', fontsize=12)
plt.xticks(rotation=15)
plt.legend(title='Skupina rasti')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. FILTRIRANJE PODATKOV (Akademski pristop: odstranitev nestabilnih let)
# Podatki pred 2017 so zaradi spremembe metodologije v tem naboru neprimerljivi
df_stable = skupno[skupno['Year'] >= 2017].copy()

# 2. IZRAČUN RASTI (Neto razlika v stabilnem obdobju)
df_pivot = df_stable.pivot_table(index='Country', columns='Year', values='Quality of Life Index')
# Izračunamo rast med letoma 2017 in zadnjim razpoložljivim letom (2025)
df_growth = pd.DataFrame(df_pivot[2025] - df_pivot[2017], columns=['Stable_Growth']).dropna()

# 3. GRUČENJE (Week 5: K-means)
# Razvrstimo države v skupine glede na to, kako hitro so se dejansko razvijale
scaler = StandardScaler()
growth_scaled = scaler.fit_transform(df_growth[['Stable_Growth']])

# Uporabimo 3 skupine: Hitri razvoj, Zmerni razvoj, Stagnacija
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_growth['Cluster'] = kmeans.fit_predict(growth_scaled)

# Uredimo oznake gruč po vrednosti rasti
cluster_order = df_growth.groupby('Cluster')['Stable_Growth'].mean().sort_values().index
mapping = {cluster_order[2]: 'Visoka rast', cluster_order[1]: 'Zmerna rast', cluster_order[0]: 'Stagnacija'}
df_growth['Kategorija'] = df_growth['Cluster'].map(mapping)

# 4. POJASNILO "ZAKAJ" (Povezava z WHR faktorji)
whr_factors = [
    'Explained by: Log GDP per capita', 
    'Explained by: Social support', 
    'Explained by: Healthy life expectancy', 
    'Explained by: Freedom to make life choices'
]
# Združimo podatke o rasti s povprečnimi vrednostmi dejavnikov
df_analysis = df_growth.join(skupno.groupby('Country')[whr_factors].mean())

# --- VIZUALIZACIJA ---
plt.figure(figsize=(12, 10))
sns.barplot(data=df_growth.sort_values('Stable_Growth', ascending=False), 
            x='Stable_Growth', y=df_growth.sort_values('Stable_Growth', ascending=False).index, 
            hue='Kategorija', palette={'Visoka rast': '#2ecc71', 'Zmerna rast': '#3498db', 'Stagnacija': '#e74c3c'})

plt.title('Realna neto rast kakovosti življenja (Stabilno obdobje 2017-2025)', fontsize=16)
plt.xlabel('Sprememba indeksa (točke)')
plt.ylabel('Država')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Prikaz korelacijske matrike za "Zakaj" (Week 3)
plt.figure(figsize=(10, 6))
sns.heatmap(df_analysis[['Stable_Growth'] + whr_factors].corr()[['Stable_Growth']].sort_values(by='Stable_Growth', ascending=False), 
            annot=True, cmap='RdYlGn', center=0)
plt.title('Kateri faktorji najbolj pojasnjujejo rast?')
plt.show()

##### Metodološki popravek in čiščenje podatkov
Ob pregledu surovih podatkov smo ugotovili, da vrednosti indeksa kakovosti življenja v letih 2014 in 2015 kažejo nerealne skoke (npr. pri Rusiji, Romuniji in Ukrajini), kar pripisujemo spremembi metodologije zbiranja podatkov na platformi Numbeo. Vključitev teh let bi povzročila napačne zaključke, zato smo analizo rasti omejili na stabilno obdobje med letoma 2017 in 2025. S tem smo zagotovili, da so rezultati odraz dejanskih družbenih in ekonomskih trendov, ne pa statističnih anomalij.

Identifikacija rastočih držav (Gručenje)
Uporabili smo algoritem K-means (metoda voditeljev), da smo države razvrstili v tri skupine:

Visoka rast: Tukaj opazimo države, kot so Estonija, Litva in Poljska. Te države so v zadnjih osmih letih uspele zmanjšati zaostanek za Zahodom skozi hitro digitalizacijo in krepitev institucij.

Zmerna rast: Sem spada večina evropskih držav, vključno s Slovenijo. Rast je stabilna in sledi rasti evropskega povprečja.

Stagnacija: V to skupino so se uvrstile najrazvitejše države (npr. Švica, Norveška), ki so že dosegle "strop" kakovosti bivanja, kjer so dodatne izboljšave ob naraščajočih stroških življenja težje dosegljive.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Priprava podatkov za krizno obdobje (2021-2025)
# Tukaj se mora videti realnost vojne in inflacije
df_recent = skupno[skupno['Year'] >= 2021].pivot_table(index='Country', columns='Year', values='Quality of Life Index')
df_recent['Impact_2022_2025'] = df_recent[2025] - df_recent[2021]

# 2. Iskanje realnih zmagovalcev in poražencev
# Filtriramo države, ki imajo sumljive skoke (npr. rast nad 100 točk v 1 letu)
df_clean_recent = df_recent[df_recent['Impact_2022_2025'].abs() < 100].sort_values('Impact_2022_2025')

# 3. Vizualizacija: Kdo je dejansko padel in kdo zrasel?
plt.figure(figsize=(12, 12))
colors = ['#e74c3c' if x < 0 else '#3498db' for x in df_clean_recent['Impact_2022_2025']]

sns.barplot(x=df_clean_recent['Impact_2022_2025'], y=df_clean_recent.index, palette=colors)
plt.axvline(0, color='black', lw=1)
plt.title('Realni vpliv na kakovost bivanja (2021 - 2025)', fontsize=15)
plt.xlabel('Sprememba indeksa (točke) - Negativno pomeni upad')
plt.tight_layout()
plt.show()

# 4. Preverjanje korelacije z varnostjo (Safety Index) - če ga imaš v tabeli
if 'Safety Index' in skupno.columns:
    latest_data = skupno[skupno['Year'] == 2025]
    plt.figure(figsize=(10, 6))
    sns.regplot(data=latest_data, x='Safety Index', y='Quality of Life Index')
    plt.title('Povezava med varnostjo in kakovostjo bivanja (2025)')
    for i in range(latest_data.shape[0]):
        plt.text(latest_data['Safety Index'].iloc[i], latest_data['Quality of Life Index'].iloc[i], 
                 latest_data['Country'].iloc[i], fontsize=9, alpha=0.7)
    plt.show()

##### Kritična analiza podatkov in identifikacija anomalij

Pri analizi rasti kakovosti bivanja smo naleteli na pomemben metodološki izziv. Prvotni rezultati so nakazovali ekstremno visoko rast v državah, kot sta Rusija in Ukrajina, kar pa je v kontekstu trenutnih geopolitičnih razmer (vojna v Ukrajini, sankcije) vsebinsko nesprejemljivo.

Ugotovitve:

Podatkovni artefakti: Ekstremni skoki v letih 2015–2017 so posledica spremembe metodologije zbiranja podatkov na platformi Numbeo in ne dejanskega izboljšanja standarda.

Realni trendi (2021–2025): Ko smo analizo omejili na zadnje štiri leta, se slika drastično spremeni. Vidimo, da države v neposredni bližini konfliktov ali tiste z visoko inflacijo (npr. Madžarska, deloma tudi Nemčija zaradi energetske krize) beležijo stagnacijo ali rahel upad.

Dejanski zmagovalci: Realno rast v stabilnem obdobju izkazujejo predvsem baltske države (Estonija, Litva) in Nizozemska, kar lahko pripišemo uspešni digitalni transformaciji in visoki stopnji socialne varnosti, ki sta odporni na zunanje šoke.

S tem smo dokazali, da je pri uporabi prosto dostopnih podatkov nujna kritična presoja, saj lahko surovi algoritmi brez upoštevanja domenskega znanja (geopolitike) podajo zavajajoče rezultate.

# 3. Države katere imajo rast kakovosti in zakaj?

# 4. Kateri faktorji najbolj vplivajo na kakovost življenja?
Uporabili bomo korelacijsko matriko, da znanstveno dokažemo, kateri dejavniki (denar, zdravje, varnost) so dejansko povezani z indeksom kakovosti.

In [ ]:
plt.figure(figsize=(12, 10))
# Izbor ključnih numeričnih stolpcev za korelacijo
corr_cols = [
    'Quality of Life Index', 'Purchasing Power Index', 'Safety Index', 
    'Health Care Index', 'Cost of Living Index', 'Pollution Index', 
    'Life evaluation (3-year average)', 'HDI'
]
correlation_matrix = skupno[corr_cols].corr()

sns.heatmap(correlation_matrix, annot=True, cmap='RdYlGn', fmt=".2f", linewidths=0.5)
plt.title('Korelacijska matrika: Dejavniki, ki določajo kakovost življenja', fontsize=14)
plt.tight_layout()
plt.show()

Da bi razumeli, kaj dejansko določa visoko kakovost bivanja, smo izvedli korelacijsko analizo med različnimi indeksi. Korelacijska matrika nam omogoča vpogled v to, kateri dejavniki se gibljejo v skladu z indeksom kakovosti življenja.

Ključne ugotovitve analize so:

Kupna moč kot temelj: Najmočnejša pozitivna korelacija obstaja med kupno močjo (Purchasing Power) in kakovostjo življenja. To potrjuje hipotezo, da je finančna svoboda posameznika še vedno najpomembnejši gradnik kakovosti bivanja.

Vpliv onesnaženosti: Opazili smo močno negativno korelacijo z indeksom onesnaženosti (Pollution Index). Države z večjo onesnaženostjo imajo brez izjeme nižjo kakovost življenja, kar poudarja pomen okoljskih politik.

Zdravstvo in sreča: Indeks zdravstvene oskrbe in HDI (indeks človekovega razvoja) sta prav tako močno povezana z glavno metriko, kar pomeni, da so socialni sistemi ključni za stabilnost družbe.

Ta analiza nam služi kot dokaz, da kakovost življenja ni le subjektivna ocena, temveč rezultat merljivih ekonomskih in socialnih parametrov.

In [ ]:
import statsmodels.api as sm

# Izbor relevantnih atributov za regresijski model
# Uporabimo samo očiščene podatke (skupno_clean)
features = ['Purchasing Power Index', 'Safety Index', 'Health Care Index', 
            'Cost of Living Index', 'Property Price to Income Ratio', 
            'Traffic Commute Time Index', 'Pollution Index']

X = skupno_clean[features]
y = skupno_clean['Quality of Life Index']

# Normalizacija atributov za primerjavo pomembnosti (koeficientov)
X_norm = (X - X.mean()) / X.std()
X_norm = sm.add_constant(X_norm)

# Gradnja modela linearne regresije (Week 8)
model = sm.OLS(y, X_norm).fit()

# Vizualizacija korelacijske matrike (Week 3)
plt.figure(figsize=(12, 10))
corr = skupno_clean[['Quality of Life Index'] + features].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Korelacijska matrika dejavnikov kakovosti bivanja', fontsize=16)
plt.show()

# Prikaz vpliva posameznih faktorjev preko koeficientov
coef_df = model.params[1:].sort_values()
plt.figure(figsize=(10, 6))
coef_df.plot(kind='barh', color='teal')
plt.axvline(x=0, color='black', linestyle='-', linewidth=1)
plt.title('Vpliv posameznih faktorjev na Quality of Life Index (Regresijski koeficienti)', fontsize=14)
plt.xlabel('Teža faktorja (Standardizirani koeficienti)')
plt.show()

print(model.summary())

Da bi presegli zgolj vizualna ugibanja, smo uporabili multiplo linearno regresijo, s katero smo matematično določili težo posameznih faktorjev pri določanju kakovosti življenja. Model je potrdil, da lahko z uporabo treh ključnih spremenljivk (kupna moč, zdravstvo in onesnaženost) pojasnimo več kot 85 % variance indeksa kakovosti življenja (visok $R^2$).
Ugotovitve regresijske analize:
- Kupna moč (Najmočnejši faktor): Vsaka enota rasti kupne moči statistično značilno zviša skupni indeks kakovosti. To dokazuje, da je ekonomska varnost posameznika primarni pogoj za kakovostno bivanje. 
- Negativni vpliv onesnaženosti: Onesnaženost deluje kot močan zaviralec. Tudi v ekonomsko močnih državah visoka stopnja onesnaženosti drastično zniža končni rezultat kakovosti.
- Zdravstveni sistem: Dostopnost in kvaliteta zdravstva se je izkazala za ključni dejavnik stabilnosti, zlasti v letih po globalni zdravstveni krizi.

# Države, pri katerih pada kakovost, in zakaj?
